In [11]:
%pip install pypdf
%pip install ollama


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
# CV PARSER

PROFILE_PROMPT = """
Analyze the following CV and extract a structured Python dictionary
with EXACTLY this schema:

{{
  "seniority": string,
  "years_experience": integer,
  "roles": [string],
  "core_skills": [string],
  "secondary_skills": [string],
  "certifications": [string],
  "visa_status": {{
    "eu": "yes",
    "usa": "yes"
  }}
}}

Rules:
- seniority must be one of: Junior, Mid, Senior, Staff, Principal
- core_skills = main technologies used professionally
- secondary_skills = tools mentioned but not core
- Normalize skills to lowercase
- Certifications must be official names
- If visa info is not present, use null
- Return ONLY valid JSON

CV TEXT:
----------------
{cv_text}
----------------
"""

from pypdf import PdfReader
from ollama import Client
import json
import configparser
import os

def load_config():
    config_file = 'config.ini'
    config = configparser.ConfigParser()
    config.read(config_file)
    if os.path.exists(config_file):
            with open(config_file, 'r', encoding='utf-8') as f:
                config.read_file(f)
                host = config.get("server","host")
                port = config.get("server","port")
                return "http://{host}:{port}".format(host=host, port=port)

def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    return "\n".join(page.extract_text() for page in reader.pages if page.extract_text())


def parse_cv_with_llm(cv_text: str):
    host = load_config()
    print("Using bot server at:", host)
    client = Client(host=host)
    response = client.chat(
                    model='gemma3:1b',
                    messages=[
                {"role": "system", "content": "You are an expert HR and technical CV parser."
                "You extract structured data from CVs."},
                {
                    "role": "user",
                    "content": PROFILE_PROMPT.format(cv_text=cv_text)
                }
            ],
                    format='json')
    return json.loads(response["message"]["content"])


print("Loading CV and extracting text...")

cv_text = extract_text_from_pdf("CV_JerrySanchez_English_v5.pdf")
profile = parse_cv_with_llm(cv_text)

#print("MY_PROFILE = ")
#print(json.dumps(profile, indent=4))
with open('profile.json', 'w') as f:
    json.dump(profile, f, indent=4)


Loading CV and extracting text...
Using bot server at: http://127.0.0.1:11434
